In [1]:
import feedparser
from openai import OpenAI
import datetime
from dateutil import parser as dateparser
import json
import requests
import os
import re # Import regex for creating anchor slugs
from bs4 import BeautifulSoup # For cleaning HTML from summaries
from dotenv import load_dotenv

# ------------- CONFIG ----------------

# Load environment variables from .env file (must be in the same directory)
load_dotenv()
   
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
  
if not OPENAI_API_KEY:
	raise ValueError("OPENAI_API_KEY not found. Make sure it's set in your .env file.")

# Use the modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Using gpt-4o-mini: faster, more capable, and cost-effective for this task.
LLM_MODEL = "gpt-4o-mini"

# --- User-Curated RSS Feeds ---
RSS_FEEDS = {
    # NEP - Economics
    "NEP-FOR": "https://nep.repec.org/rss/nep-for.rss.xml",   # **Forecasting** (Most Important)

}


SCORE_THRESHOLD = 8.0  # Keep only papers scoring above this
DAYS_BACK = 45         # Look back this many days
OUTPUT_DIR = "newsletters"

# ------------- HELPER FUNCTIONS ----------------

def _get_authors(entry):
    """Normalizes author information from a feed entry."""
    if hasattr(entry, 'authors') and entry.authors:
        return ', '.join(author['name'] for author in entry.authors if 'name' in author)
    if hasattr(entry, 'author'):
        return entry.author
    return "Unknown"

def _create_anchor_slug(title, index):
    """Creates a URL-friendly slug from a title for anchor links."""
    s = title.lower()
    s = re.sub(r'[^\w\s-]', '', s) # Remove non-alphanumeric characters
    s = re.sub(r'[\s_-]+', '-', s).strip('-') # Replace spaces with hyphens
    return f"{s}-{index}"

# ------------- CORE FUNCTIONS ----------------

def fetch_recent_papers():
    """Fetch recent papers from RSS feeds with robust date parsing and encoding correction."""
    print(f"Fetching papers from {len(RSS_FEEDS)} sources, looking back {DAYS_BACK} days...")
    cutoff_date = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=DAYS_BACK)
    papers = []

    for source, url in RSS_FEEDS.items():
        print(f"  - Processing {source} from {url}")
        try:
            response = requests.get(url, timeout=15)
            if "utf-8" in response.text.lower() or "<?xml" in response.text:
                response.encoding = "utf-8"
            feed = feedparser.parse(response.text)
            if feed.bozo:
                print(f"    Warning: Malformed feed for {source}. Error: {feed.bozo_exception}")

            found_in_feed = 0
            for entry in feed.entries:
                pub_date_str = getattr(entry, "published", getattr(entry, "updated", None))
                if not pub_date_str: continue
                try:
                    pub_date = dateparser.parse(pub_date_str)
                except dateparser.ParserError:
                    continue
                if pub_date.tzinfo is None:
                    pub_date = pub_date.replace(tzinfo=datetime.timezone.utc)
                if pub_date < cutoff_date: continue
                summary = getattr(entry, "summary", "")
                if "<" in summary and ">" in summary:
                    summary = BeautifulSoup(summary, "html.parser").get_text(separator=' ', strip=True)
                papers.append({
                    "title": entry.title, "link": entry.link, "summary": summary,
                    "authors": _get_authors(entry), "source": source, "date": pub_date.strftime("%Y-%m-%d")
                })
                found_in_feed += 1
            print(f"    Found {found_in_feed} recent papers in {source}.")
        except Exception as e:
            print(f"    Error fetching or parsing feed {source}: {e}")
            continue
    return papers

def score_and_summarize(paper):
    """Sends abstract to LLM for summary, scoring, and categorization."""
    prompt = f"""
You are a highly specialized expert curator for "ets4 Monthly (Economic Time Series Forecasting Monthly)," a newsletter focused **exclusively on practical and impactful forecasting of economic time series.** Your role is to identify only the most relevant and innovative work for an audience of researchers and practitioners.

You will be given a research paper. Evaluate it strictly through the lens of **forecasting relevance and potential to advance economic prediction.** Papers that focus on **structural analysis, causal inference, descriptive statistics, or purely theoretical econometrics without a predictive component must be classified as Not Relevant**, even if they appear in economics journals.

---

**Paper Information**

Title: {paper['title']}
Authors: {paper['authors']}
Source: {paper['source']}
Date: {paper['date']}
Abstract: {paper['summary']}

---

### Your Tasks

1. **Summarize (3-4 sentences)**  
   Focus only on elements related to **forecasting or predictive modeling**. If forecasting is not clearly present, explicitly say so.

2. **Assign a Quality Score (1–10)**  
   Score based on **forecasting methodological novelty, empirical rigor, and potential practical impact.**  
   - **Extra weight should be given** to papers that:
     - Introduce **new modeling approaches** or **improve existing ones significantly**.
     - Use **novel datasets** that could meaningfully enhance economic forecasting.
     - Present **forecasting applications with real-world decision value** (e.g. monetary policy, energy prices, financial trading, inflation nowcasting).

3. **Classify into One Category**  
   Choose exactly one:
   - **Directly Relevant** → The paper is clearly about **forecasting *economic* time series** and offers meaningful contributions or applications.
   - **Paper of Interest** → The paper is *not economic*, but presents **highly innovative forecasting methods or data strategies** that could plausibly transfer to economics.
   - **Not Relevant** → No clear predictive component, or relevance is too indirect (e.g., **structural models, policy simulations without forecasting, causal effects estimation, variance decompositions, statistical/econometric theory without prediction focus**).

---

### Scoring & Categorization Rules

| Score | Meaning | Default Category |
|--------|---------|------------------|
| **1–3** | Low quality OR no forecasting at all | Not Relevant |
| **4–6** | Minor/incremental forecasting contribution OR niche application with little general value | Not Relevant |
| **7–8** | Solid contribution to forecasting | Directly Relevant *if economic*, otherwise Paper of Interest |
| **9–10** | Breakthrough forecasting methodology, dataset, or application with clear future impact | Directly Relevant *if economic*, otherwise Paper of Interest |

---

### JSON Output Format

Return your response strictly in **valid JSON** using the following schema:

{{
"summary": "...",
"score": X,
"category": "Directly Relevant" | "Paper of Interest" | "Not Relevant",
"adaptability_reason": null | "Required only if category is 'Paper of Interest'"
}}

- If you select **Paper of Interest**, you **must** fill in `"adaptability_reason"` with a short statement explaining how the method could be applied to economic forecasting.
- For **Directly Relevant** or **Not Relevant**, set `"adaptability_reason": null`.
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        data = json.loads(content)
        required_keys = ["summary", "score", "category", "adaptability_reason"]
        if not all(key in data for key in required_keys):
            raise ValueError(f"LLM response missing one or more required keys: {required_keys}")
        if not isinstance(data["score"], (int, float)):
            raise ValueError("LLM score is not a number.")
        if data["category"] == "Paper of Interest" and not data.get("adaptability_reason"):
            raise ValueError("Category is 'Paper of Interest' but adaptability_reason is missing.")
        data["score"] = float(data["score"])
        return data
    except Exception as e:
        print(f"Error during LLM call for paper '{paper['title']}': {e}")
        return {"summary": f"Error: {e}", "score": 0.0, "category": "Not Relevant", "adaptability_reason": None}
    
def _create_anchor_slug(title):
    """Creates a Markdown-compatible anchor slug from a title."""
    s = title.lower()
    s = re.sub(r'[^\w\s-]', '', s) # Remove non-alphanumeric characters
    s = re.sub(r'[\s_]+', '-', s).strip('-') # Replace spaces with a single hyphen
    return s

def build_internal_markdown(relevant_papers, interest_papers, filename):
    """Creates the internal markdown file WITH scores for review."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today_str = datetime.date.today().strftime('%Y-%m-%d')
    lines = [f"# ets4 Monthly [INTERNAL] – {today_str}\n", "Internal review copy with quality scores.\n"]
    lines.append("## Directly Relevant Papers\n")
    if not relevant_papers:
        lines.append("No directly relevant papers met the criteria this month.\n")
    else:
        relevant_papers.sort(key=lambda p: p['score'], reverse=True)
        for p in relevant_papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Quality Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}\n")
    lines.append("\n---\n")
    lines.append("## Papers of Interest\n")
    if not interest_papers:
        lines.append("No papers of interest were identified this month.\n")
    else:
        interest_papers.sort(key=lambda p: p['score'], reverse=True)
        for p in interest_papers:
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Quality Score:** {p['score']:.1f}/10")
            lines.append(f"- **Summary:** {p['summary']}")
            lines.append(f"- **Reason for Interest:** {p.get('adaptability_reason', 'N/A')}\n")
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✅ Internal review file saved: {filename}")

def build_public_markdown(relevant_papers, interest_papers, filename):
    """Creates a public-facing markdown file with a correctly numbered, Markdown-native Table of Contents."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today = datetime.date.today()
    
    iso_date_str = today.strftime('%Y-%m-%d')
    title_date_str = today.strftime('%B %d, %Y')

    front_matter = f"""---
title: "ets4 Newsletter, {title_date_str}"
date: {iso_date_str}
draft: true
---
"""
    lines = [
        front_matter,
        f"# ets4 Monthly – {iso_date_str}\n",
        "A curated list of recent papers on novel methods for forecasting economic time series.\n"
    ]

    # --- Deduplicate and Sort Papers ---
    seen_titles = set()
    unique_relevant = []
    for p in sorted(relevant_papers, key=lambda x: x['score'], reverse=True):
        normalized_title = p['title'].lower().strip()
        if normalized_title not in seen_titles:
            unique_relevant.append(p)
            seen_titles.add(normalized_title)

    unique_interest = []
    for p in sorted(interest_papers, key=lambda x: x['score'], reverse=True):
        normalized_title = p['title'].lower().strip()
        if normalized_title not in seen_titles:
            unique_interest.append(p)
            seen_titles.add(normalized_title)

    # --- Build Table of Contents ---
    if unique_relevant:
        lines.append("## Directly Relevant Papers\n")
        lines.append("### In this Issue\n")
        for i, p in enumerate(unique_relevant):
            # Use the new, simpler slug function
            anchor = _create_anchor_slug(p['title'])
            lines.append(f"{i + 1}. [{p['title']} - *{p['authors']}*](#{anchor})")
        lines.append("")

    if unique_interest:
        if not unique_relevant:
             lines.append("## Papers of Interest\n")
        else:
            lines.append("\n---\n")
            lines.append("## Papers of Interest\n")
        lines.append("_Methodologically novel papers from other fields that could be adapted for economic forecasting._\n")
        lines.append("### In this Issue\n")
        for i, p in enumerate(unique_interest):
            # Use the new, simpler slug function
            anchor = _create_anchor_slug(p['title'])
            lines.append(f"{i + 1}. [{p['title']} - *{p['authors']}*](#{anchor})")
        lines.append("")

    # --- Build Full Paper Details ---
    if unique_relevant:
        lines.append("\n---\n")
        lines.append("### Paper Details\n")
        for p in unique_relevant:
            # --- FINAL FIX: Use a standard Markdown header. The anchor is generated automatically. ---
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Summary:** {p['summary']}\n")
    
    if unique_interest:
        if unique_relevant:
            lines.append("\n---\n")
        lines.append("### Paper Details\n")
        for p in unique_interest:
            # --- FINAL FIX: Use a standard Markdown header. The anchor is generated automatically. ---
            lines.append(f"### [{p['title']}]({p['link']})")
            lines.append(f"- **Authors:** {p['authors']}")
            lines.append(f"- **Source:** {p['source']} ({p['date']})")
            lines.append(f"- **Summary:** {p['summary']}")
            lines.append(f"- **Reason for Interest:** {p.get('adaptability_reason', 'N/A')}\n")

    if not unique_relevant and not unique_interest:
        lines.append("## No papers met the criteria this month. Check back soon!\n")
        
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✅ Public newsletter file saved: {filename}")


# ------------- MAIN ----------------

if __name__ == "__main__":
    print("Starting Economic Forecasting Newsletter Pipeline...")
    all_papers = fetch_recent_papers()
    print(f"Found {len(all_papers)} total recent papers.")
    unique_papers = []
    seen_links = set()
    for paper in all_papers:
        if paper['link'] not in seen_links:
            unique_papers.append(paper)
            seen_links.add(paper['link'])
    if len(all_papers) > len(unique_papers):
        print(f"Removed {len(all_papers) - len(unique_papers)} link-based duplicates. Processing {len(unique_papers)} unique papers.")
    
    shortlisted_relevant = []
    shortlisted_interest = []
    for i, paper in enumerate(unique_papers, 1):
        print(f"Processing paper {i}/{len(unique_papers)}: '{paper['title'][:70]}...'")
        result = score_and_summarize(paper)
        paper.update(result)
        if paper.get("score", 0.0) >= SCORE_THRESHOLD:
            if paper.get("category") == "Directly Relevant":
                shortlisted_relevant.append(paper)
                print(f"  -> Shortlisted! (Relevant) Score: {paper['score']:.1f}/10")
            elif paper.get("category") == "Paper of Interest":
                shortlisted_interest.append(paper)
                print(f"  -> Shortlisted! (Interest) Score: {paper['score']:.1f}/10")
            else:
                print(f"  -> Skipped. Category: {paper.get('category', 'Unknown')}, Score: {paper['score']:.1f}/10")
        else:
            print(f"  -> Skipped. Score below threshold: {paper.get('score', 0.0):.1f}/10")
    
    today_str = datetime.date.today().strftime("%Y-%m-%d")
    internal_filename = os.path.join(OUTPUT_DIR, f"ets4_monthly_{today_str}_INTERNAL.md")
    public_filename = os.path.join(OUTPUT_DIR, f"ets4_monthly_{today_str}_PUBLIC.md")
    
    build_internal_markdown(shortlisted_relevant, shortlisted_interest, internal_filename)
    build_public_markdown(shortlisted_relevant, shortlisted_interest, public_filename)
    
    print("\n--- Pipeline Finished ---")
    print(f"Total unique papers processed by LLM: {len(unique_papers)}")
    print(f"Shortlisted {len(shortlisted_relevant)} 'Directly Relevant' papers.")
    print(f"Shortlisted {len(shortlisted_interest)} 'Papers of Interest'.")

Starting Economic Forecasting Newsletter Pipeline...
Fetching papers from 1 sources, looking back 45 days...
  - Processing NEP-FOR from https://nep.repec.org/rss/nep-for.rss.xml
    Found 10 recent papers in NEP-FOR.
Found 10 total recent papers.
Processing paper 1/10: 'Local Estimation for Option Pricing: Improving Forecasts with Market S...'
  -> Shortlisted! (Relevant) Score: 8.0/10
Processing paper 2/10: 'Adaptive Temporal Fusion Transformers for Cryptocurrency Price Predict...'
  -> Shortlisted! (Interest) Score: 9.0/10
Processing paper 3/10: 'Out-of-sample gravity predictions and trade policy counterfactuals...'
  -> Shortlisted! (Relevant) Score: 8.0/10
Processing paper 4/10: 'Stabilising Lifetime PD Models under Forecast Uncertainty...'
  -> Shortlisted! (Relevant) Score: 8.0/10
Processing paper 5/10: 'Meta-Learning Neural Process for Implied Volatility Surfaces with SABR...'
  -> Shortlisted! (Relevant) Score: 8.0/10
Processing paper 6/10: 'When Tails Are Heavy: The Benefits 